### DATA Prepping

In [1]:
!pip install --quiet presidio-analyzer
!pip --quiet install spacy
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     -- ------------------------------------- 0.8/12.8 MB 3.0 MB/s eta 0:00:04
     ---- ----------------------------------- 1.6/12.8 MB 2.8 MB/s eta 0:00:05
     ------- -------------------------------- 2.4/12.8 MB 3.1 MB/s eta 0:00:04
     ----------- ---------------------------- 3.7/12.8 MB 3.8 MB/s eta 0:00:03
     -------------- ------------------------- 4.7/12.8 MB 4.1 MB/s eta 0:00:02
     ------------------ --------------------- 5.8/12.8 MB 4.2 MB/s eta 0:00:02
     ---------------------- ----------------- 7.1/12.8 MB 4.5 MB/s eta 0:00:02
     ------------------------- -------------- 8.1/12.8 MB 4.6 MB/s eta 0:00:02
     ----------------------------- ---------- 9.4/12.8 MB 4.7 MB/s eta 0:00:01
     -------------------------------- ------- 10.5/12.8 MB 4.8 MB/s eta 0:00:01
     ------------------------------------ --- 11.5/12.8 MB 4.8 MB

In [2]:
import json
import random
import pickle
import matplotlib.pyplot as plt
from tqdm import tqdm
import spacy
from spacy.training.example import Example
from spacy.util import minibatch
from presidio_analyzer import AnalyzerEngine

In [3]:
from utils import *

In [4]:
df_full = pd.read_csv('PII43k.csv', on_bad_lines='skip')

df_full.head()

,Template,Filled Template,Tokenised Filled Template,Tokens
0,"In our video conference, discuss the role of e...","In our video conference, discuss the role of e...","['in', 'our', 'video', 'conference', ',', 'dis...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
1,Could you draft a letter for [NAME_1] to send ...,"Could you draft a letter for Dietrich, Schulis...","['could', 'you', 'draft', 'a', 'letter', 'for'...","['O', 'O', 'O', 'O', 'O', 'O', 'B-NAME', 'I-NA..."
2,Discuss the options for [FULLNAME_1] who wants...,Discuss the options for Jeffery Pfeffer who wa...,"['discuss', 'the', 'options', 'for', 'jeff', '...","['O', 'O', 'O', 'O', 'B-FULLNAME', 'I-FULLNAME..."
3,13. Write a press release announcing [FULLNAME...,13. Write a press release announcing Gayle Wat...,"['13', '.', 'write', 'a', 'press', 'release', ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-FULLNAM..."
4,9. Develop an inventory management plan for [F...,9. Develop an inventory management plan for Ev...,"['9', '.', 'develop', 'an', 'inventory', 'mana...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-FU..."


In [5]:
cleaned_matches, unique_matches = get_template_tokens(df_full)

def replace_unique_tokens(text, tokens):
	for token in tokens:
		# Match either an underscore with one or more digits or with 'N'
		pattern = r'\[' + token + r'_(?:\d+|N)\]'
		# Replace with the token in square brackets (e.g., "[NAME]")
		text = re.sub(pattern, f'[{token}]', text)
	return text

df_full['Template'] = df_full['Template'].apply(lambda t: replace_unique_tokens(t, cleaned_matches))

# view the first 5 rows of the 'Template' column
df_full['Template'].head()

0    In our video conference, discuss the role of e...
1    Could you draft a letter for [NAME] to send to...
2    Discuss the options for [FULLNAME] who wants t...
3    13. Write a press release announcing [FULLNAME...
4    9. Develop an inventory management plan for [F...
Name: Template, dtype: object

In [6]:
import pandas as pd
import re


list_of_weird_rows = [36484,42606,39778,39591,38894,35859,35398,34377,33200,32786,32633,31525,31171,30957,30419,30129,28097,25492,25335,24531,24497,23920,22670,22152,19842,19644,19461,18718,16604,16297,16239,15847,15775,15549,14604,13642,13103,12358,11358,10344,8652,738,6801,5071,4265,3654,3381,3218,1906,38778,36440,35086,7368,13681,29797,7384,13709,29863,35170,38822,30957]

# Original row count
original_count = len(df_full)
print(f"Original number of rows: {original_count}")

# Drop the rows
df_full = df_full.drop(index=list_of_weird_rows, errors='ignore')

# Reset index to have sequential row numbers
df_full = df_full.reset_index(drop=True)
print(f"Number of rows after dropping weird rows: {len(df_full)}")
print(f"Rows removed: {original_count - len(df_full)}")

# Shuffle df_full
df_full = df_full.sample(frac=1, random_state=42).reset_index(drop=True)

Original number of rows: 42759
Number of rows after dropping weird rows: 42699
Rows removed: 60


In [7]:
df_full.head()

,Template,Filled Template,Tokenised Filled Template,Tokens
0,[FIRSTNAME] needs help designing a user-friend...,Halie needs help designing a user-friendly che...,"['hal', '##ie', 'needs', 'help', 'designing', ...","['B-FIRSTNAME', 'I-FIRSTNAME', 'O', 'O', 'O', ..."
1,Draft a letter to [NAME] explaining the import...,"Draft a letter to Runte, McGlynn and Kautzer e...","['draft', 'a', 'letter', 'to', 'run', '##te', ...","['O', 'O', 'O', 'O', 'B-NAME', 'I-NAME', 'I-NA..."
2,3. Describe the process of conducting an inter...,3. Describe the process of conducting an inter...,"['3', '.', 'describe', 'the', 'process', 'of',...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
3,Can you tell me how companies should handle cu...,Can you tell me how companies should handle cu...,"['can', 'you', 'tell', 'me', 'how', 'companies...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
4,Write a research proposal for [NAME] to study ...,Write a research proposal for Wolf - Reilly to...,"['write', 'a', 'research', 'proposal', 'for', ...","['O', 'O', 'O', 'O', 'O', 'B-NAME', 'I-NAME', ..."


In [8]:
cleaned_matches, unique_matches = get_template_tokens(df_full)
print(unique_matches)
print(len(unique_matches))

{'JOBTITLE', 'ETHEREUMADDRESS', 'LITECOINADDRESS', 'BUILDINGNUMBER', 'ZIPCODE', 'SEXTYPE', 'FULLNAME', 'SECONDARYADDRESS', 'STREET', 'USERNAME', 'USERAGENT', 'DISPLAYNAME', 'MASKEDNUMBER', 'CURRENCYSYMBOL', 'CURRENCYNAME', 'COUNTY', 'FIRSTNAME', 'STREETADDRESS', 'GENDER', 'PIN', 'BITCOINADDRESS', 'STATE', 'IBAN', 'JOBAREA', 'MAC', 'CREDITCARDISSUER', 'ORDINALDIRECTION', 'NUMBER', 'CREDITCARDNUMBER', 'ACCOUNTNAME', 'ACCOUNTNUMBER', 'NAME', 'CURRENCY', 'LASTNAME', 'URL', 'AMOUNT', 'CITY', 'EMAIL', 'SEX', 'IPV6', 'IP', 'IPV4', 'CREDITCARDCVV', 'CURRENCYCODE', 'PASSWORD', 'JOBTYPE', 'BIC', 'NEARBYGPSCOORDINATE', 'JOBDESCRIPTOR'}
49


In [9]:
# Display a few rows to verify the changes
print(df_full[['Template', 'Filled Template']].head())

                                            Template  \
0  [FIRSTNAME] needs help designing a user-friend...   
1  Draft a letter to [NAME] explaining the import...   
2  3. Describe the process of conducting an inter...   
3  Can you tell me how companies should handle cu...   
4  Write a research proposal for [NAME] to study ...   

                                     Filled Template  
0  Halie needs help designing a user-friendly che...  
1  Draft a letter to Runte, McGlynn and Kautzer e...  
2  3. Describe the process of conducting an inter...  
3  Can you tell me how companies should handle cu...  
4  Write a research proposal for Wolf - Reilly to...  


In [10]:
# Replace any occurrence of "[FULLNAME]" (with optional suffix) in the Template 
# with "[NAME] [NAME]"

df_full['Template'] = df_full['Template'].apply(
	lambda t: re.sub(r'\[FULLNAME(?:_(?:\d+|N))?\]', "[NAME] [NAME]", t)
)

# Verify the changes
print(df_full['Template'].head())

0    [FIRSTNAME] needs help designing a user-friend...
1    Draft a letter to [NAME] explaining the import...
2    3. Describe the process of conducting an inter...
3    Can you tell me how companies should handle cu...
4    Write a research proposal for [NAME] to study ...
Name: Template, dtype: object


In [11]:
# Display a few examples with the token "[FIRSTNAME]" in the Template
fname_examples = df_full[df_full['Template'].str.contains(r'\[FIRSTNAME_1\]', na=False)]

if fname_examples.empty:
	print("No rows found with the token [FIRSTNAME_1]")
else:
	for i, row in fname_examples.head(5).iterrows():
		print("Template:", row["Template"])
		print("Filled Template:", row["Filled Template"])
		print("-" * 60)

No rows found with the token [FIRSTNAME_1]


In [12]:
token_mapping = {
  "LOCATION": [ 
    "STREETADDRESS",
    "SECONDARYADDRESS",
    "BUILDINGNUMBER",
    "STREET",
    "CITY",
    "STATE",
    "COUNTY",
    "NEARBYGPSCOORDINATE",
    "ZIPCODE"
  ],
  "GENDER": [
    "SEXTYPE",
    "SEX"
  ],
  "USERNAME":[
    "DISPLAYNAME"
  ],
  "NAME": [
    "NAME",
    "FIRSTNAME",
    "LASTNAME"
  ],
  "IP": [
    "IPV4",
    "IP",
    "IPV6",
    "MAC"
  ],
  "JOB": [
    "JOBDESCRIPTOR",
    "JOBTYPE",
    "JOBTITLE",
    "JOBAREA"
  ],
  "MASKEDNUMBER": [
    "NUMBER",
    "AMOUNT",
    "ACCOUNTNUMBER",
    "CREDITCARDCVV",
    "PIN",
    "BUILDINGNUMBER"
  ]
}

In [13]:
def replace_tokens_with_category(text, mapping):
	for category, tokens in mapping.items():
		for token in tokens:
			# Replace tokens with a suffix (e.g., [CITY_1] or [CITY_N])
			pattern = r'\[' + token + r'_(?:\d+|N)\]'
			text = re.sub(pattern, f'[{category.upper()}]', text)
			# Replace tokens without a suffix (e.g., [CITY])
			pattern_no_suffix = r'\[' + token + r'\]'
			text = re.sub(pattern_no_suffix, f'[{category.upper()}]', text)
	return text

# Update the 'Template' column in df_full
df_full['Template'] = df_full['Template'].apply(lambda t: replace_tokens_with_category(t, token_mapping))
print(df_full['Template'].head())

_, unique_matches = get_template_tokens(df_full)
print(unique_matches)
print(len(unique_matches))

0    [NAME] needs help designing a user-friendly ch...
1    Draft a letter to [NAME] explaining the import...
2    3. Describe the process of conducting an inter...
3    Can you tell me how companies should handle cu...
4    Write a research proposal for [NAME] to study ...
Name: Template, dtype: object
{'ETHEREUMADDRESS', 'LITECOINADDRESS', 'USERAGENT', 'USERNAME', 'LOCATION', 'MASKEDNUMBER', 'JOB', 'CURRENCYSYMBOL', 'CURRENCYNAME', 'GENDER', 'BITCOINADDRESS', 'IBAN', 'CREDITCARDISSUER', 'ORDINALDIRECTION', 'CREDITCARDNUMBER', 'ACCOUNTNAME', 'NAME', 'CURRENCY', 'URL', 'EMAIL', 'IP', 'CURRENCYCODE', 'PASSWORD', 'BIC'}
24


In [14]:
df_full

,Template,Filled Template,Tokenised Filled Template,Tokens
0,[NAME] needs help designing a user-friendly ch...,Halie needs help designing a user-friendly che...,"['hal', '##ie', 'needs', 'help', 'designing', ...","['B-FIRSTNAME', 'I-FIRSTNAME', 'O', 'O', 'O', ..."
1,Draft a letter to [NAME] explaining the import...,"Draft a letter to Runte, McGlynn and Kautzer e...","['draft', 'a', 'letter', 'to', 'run', '##te', ...","['O', 'O', 'O', 'O', 'B-NAME', 'I-NAME', 'I-NA..."
2,3. Describe the process of conducting an inter...,3. Describe the process of conducting an inter...,"['3', '.', 'describe', 'the', 'process', 'of',...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
3,Can you tell me how companies should handle cu...,Can you tell me how companies should handle cu...,"['can', 'you', 'tell', 'me', 'how', 'companies...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
4,Write a research proposal for [NAME] to study ...,Write a research proposal for Wolf - Reilly to...,"['write', 'a', 'research', 'proposal', 'for', ...","['O', 'O', 'O', 'O', 'O', 'B-NAME', 'I-NAME', ..."
...,...,...,...,...
42694,Can you write a report on the recent developme...,Can you write a report on the recent developme...,"['can', 'you', 'write', 'a', 'report', 'on', '...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
42695,Can you compile a list of top 10 competitors i...,Can you compile a list of top 10 competitors i...,"['can', 'you', 'com', '##pile', 'a', 'list', '...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
42696,Please provide a list of the top 5 corporate g...,Please provide a list of the top 5 corporate g...,"['please', 'provide', 'a', 'list', 'of', 'the'...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
42697,Can you help me with a workplace culture asses...,Can you help me with a workplace culture asses...,"['can', 'you', 'help', 'me', 'with', 'a', 'wor...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."


In [15]:
# Count occurrences of each cleaned token in df_full
counts = {}
for token in unique_matches:
    # Adjust pattern to count tokens without numeric suffixes (e.g., "[NAME]")
    pattern = r'\[' + token + r'\]'
    counts[token] = int(df_full["Template"].dropna().str.count(pattern).sum())

# Convert to a DataFrame and sort by count for a nicer display
counts_df = pd.DataFrame(list(counts.items()), columns=['Token', 'Count']).sort_values(by='Count', ascending=False)
counts_df

,Token,Count
16,NAME,55826
4,LOCATION,12114
19,EMAIL,7913
6,JOB,3369
5,MASKEDNUMBER,1467
18,URL,996
3,USERNAME,713
20,IP,541
17,CURRENCY,345
22,PASSWORD,289


In [16]:
import re

# List of tokens to replace
remove_tokens = [
    "ORDINALDIRECTION", "ACCOUNTNAME", "CURRENCYSYMBOL",
    "CURRENCYNAME", "CURRENCY", "CURRENCYCODE",
    "CREDITCARDISSUER", "ETHEREUMADDRESS", "LITECOINADDRESS", "BITCOINADDRESS","BIC"
]

def unmask_text(text_with_mask, text_without_mask, tokens):
    general_token_pattern = r'\[([A-Z0-9_]+)(?:_(?:\d+|N))?\]'
    matches = list(re.finditer(general_token_pattern, text_with_mask))
    pattern_parts = []
    pos = 0

    for i, match in enumerate(matches):
        # Append literal text (escaped)
        literal_text = re.escape(text_with_mask[pos:match.start()])
        pattern_parts.append(literal_text)

        token_name = match.group(1)
        lookahead = ""
        if token_name in tokens:
            if i + 1 < len(matches):
                next_literal = text_with_mask[match.end():matches[i+1].start()]
                next_token = matches[i+1].group(1)
                if next_literal == "" and next_token not in tokens:
                    lookahead = "(?=\d)"
            # Use a unique group name for each occurrence
            group_name = f"{token_name}_{i}"
            pattern_parts.append(f"(?P<{group_name}>.+?){lookahead}")
        else:
            pattern_parts.append(".+?")
        pos = match.end()

    pattern_parts.append(re.escape(text_with_mask[pos:]))
    final_pattern = ''.join(pattern_parts)
    match_obj = re.fullmatch(final_pattern, text_without_mask)
    if match_obj:
        return match_obj.groupdict()
    return {}

def consolidate_extracted_values(extracted):
    """
    Convert keys like CREDITCARDISSUER_3, CREDITCARDISSUER_7 into a dictionary
    with key 'CREDITCARDISSUER' mapping to a list of values.
    """
    consolidated = {}
    for key, value in extracted.items():
        base = key.split('_')[0]
        consolidated.setdefault(base, []).append(value)
    return consolidated

def replace_tokens(text_with_mask, extracted_values):
    """
    Replace tokens in text_with_mask with corresponding extracted values.
    For tokens appearing multiple times, use a counter to replace them in order.
    """
    counters = {}
    def replacement(match):
        token_name = match.group(1)
        values = extracted_values.get(token_name, [])
        if not values:
            return ""
        idx = counters.get(token_name, 0)
        counters[token_name] = idx + 1
        return values[idx]
    token_regex = r'\[(' + '|'.join(remove_tokens) + r')(?:_(?:\d+|N))?\]'
    return re.sub(token_regex, replacement, text_with_mask)

# Example usage within your loop:
for i, row in df_full.iterrows():
    masked = row['Template']
    filled = row['Filled Template']
    extracted = unmask_text(masked, filled, remove_tokens)
    
    # Consolidate extracted values so that tokens map to lists
    if extracted:
        consolidated = consolidate_extracted_values(extracted)
        unmasked_text = replace_tokens(masked, consolidated)
    else:
        unmasked_text = masked

    if i == -1:
        print("Extracted values (consolidated):", consolidated)
        print("row:", i)
        print("Masked text:", masked)
        print("Filled text:", filled)
        print("-" * 50)
        print("Unmasked text:", unmasked_text)
        print("-" * 50)
        break

    if extracted != {}:
        print("Extracted values (consolidated):", consolidated)
        print("Masked text:", masked)
        print("Filled text:", filled)
        print("-" * 50)
        print("Unmasked text:", unmasked_text)
        print("-" * 50)
        df_full.at[i, 'Template'] = unmasked_text


Extracted values (consolidated): {'CURRENCYNAME': ['Swiss Franc']}
Masked text: What are the tax implications of receiving [MASKEDNUMBER] [CURRENCYNAME] as a gift?
Filled text: What are the tax implications of receiving 799.96 Swiss Franc as a gift?
--------------------------------------------------
Unmasked text: What are the tax implications of receiving [MASKEDNUMBER] Swiss Franc as a gift?
--------------------------------------------------
Extracted values (consolidated): {'CURRENCY': ['Zimbabwe Dollar']}
Masked text: Develop a project budget for [NAME] [NAME] taking into account the [CURRENCY] costs and resources needed.
Filled text: Develop a project budget for Jody Fisher taking into account the Zimbabwe Dollar costs and resources needed.
--------------------------------------------------
Unmasked text: Develop a project budget for [NAME] [NAME] taking into account the Zimbabwe Dollar costs and resources needed.
--------------------------------------------------
Extracted values

In [17]:
df_full.head()

,Template,Filled Template,Tokenised Filled Template,Tokens
0,[NAME] needs help designing a user-friendly ch...,Halie needs help designing a user-friendly che...,"['hal', '##ie', 'needs', 'help', 'designing', ...","['B-FIRSTNAME', 'I-FIRSTNAME', 'O', 'O', 'O', ..."
1,Draft a letter to [NAME] explaining the import...,"Draft a letter to Runte, McGlynn and Kautzer e...","['draft', 'a', 'letter', 'to', 'run', '##te', ...","['O', 'O', 'O', 'O', 'B-NAME', 'I-NAME', 'I-NA..."
2,3. Describe the process of conducting an inter...,3. Describe the process of conducting an inter...,"['3', '.', 'describe', 'the', 'process', 'of',...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
3,Can you tell me how companies should handle cu...,Can you tell me how companies should handle cu...,"['can', 'you', 'tell', 'me', 'how', 'companies...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
4,Write a research proposal for [NAME] to study ...,Write a research proposal for Wolf - Reilly to...,"['write', 'a', 'research', 'proposal', 'for', ...","['O', 'O', 'O', 'O', 'O', 'B-NAME', 'I-NAME', ..."


In [18]:
id = 2375
masked = df_full["Template"][id]
filled = df_full["Filled Template"][id]
extracted_values = unmask_text(masked, filled, remove_tokens)
unmasked_text = replace_tokens(masked, extracted_values)

print("Template:")
print(masked)
print("\nFilled Template:")
print(filled)

Template:
I'm curious, what's the market share of [NAME] compared to [NAME]?

Filled Template:
I'm curious, what's the market share of Maggio - Rogahn compared to Bayer, Kessler and Effertz?


In [19]:
id = 39654
masked = df_full["Template"][id]
filled = df_full["Filled Template"][id]
extracted_values = unmask_text(masked, filled, remove_tokens)
unmasked_text = replace_tokens(masked, extracted_values)

print("Template:")
print(masked)
print("\nFilled Template:")
print(filled)

Template:
I'd like to send out an email to [EMAIL] about the upcoming corporate training session. Can you help me write it?

Filled Template:
I'd like to send out an email to Nella.Lindgren@hotmail.com about the upcoming corporate training session. Can you help me write it?


In [20]:
id = 38860
masked = df_full["Template"][id]
filled = df_full["Filled Template"][id]
extracted_values = unmask_text(masked, filled, remove_tokens)
unmasked_text = replace_tokens(masked, extracted_values)

print("Template:")
print(masked)
print("\nFilled Template:")
print(filled)

Template:
Could you explain the four main themes of existential therapy mentioned in this article I found at [URL]?

Filled Template:
Could you explain the four main themes of existential therapy mentioned in this article I found at https://yummy-glasses.org?


In [21]:
# Count the labeled items in the DataFrame
total_labeled_items = count_labeled_items(df_full)
print(f"Total number of labeled items: {total_labeled_items}")
print("Number of rows in df_full:", df_full.shape[0])

Total number of labeled items: 83834
Number of rows in df_full: 42699


In [22]:
df_full.to_csv("ourdata.csv", index=False)

## AWS Comprehend dataprepping START


In [23]:
# AWS Comprehend dataprepping
aws_prepping = df_full.copy()
aws_prepping = aws_prepping.drop(columns=['Tokenised Filled Template', 'Tokens'] )
aws_prepping

,Template,Filled Template
0,[NAME] needs help designing a user-friendly ch...,Halie needs help designing a user-friendly che...
1,Draft a letter to [NAME] explaining the import...,"Draft a letter to Runte, McGlynn and Kautzer e..."
2,3. Describe the process of conducting an inter...,3. Describe the process of conducting an inter...
3,Can you tell me how companies should handle cu...,Can you tell me how companies should handle cu...
4,Write a research proposal for [NAME] to study ...,Write a research proposal for Wolf - Reilly to...
...,...,...
42694,Can you write a report on the recent developme...,Can you write a report on the recent developme...
42695,Can you compile a list of top 10 competitors i...,Can you compile a list of top 10 competitors i...
42696,Please provide a list of the top 5 corporate g...,Please provide a list of the top 5 corporate g...
42697,Can you help me with a workplace culture asses...,Can you help me with a workplace culture asses...


In [24]:
# Count occurrences of each cleaned token in df_full
counts = {}
for token in unique_matches:
    # Adjust pattern to count tokens without numeric suffixes (e.g., "[NAME]")
    pattern = r'\[' + token + r'\]'
    counts[token] = int(aws_prepping["Template"].dropna().str.count(pattern).sum())

# Convert to a DataFrame and sort by count for a nicer display
counts_df = pd.DataFrame(list(counts.items()), columns=['Token', 'Count']).sort_values(by='Count', ascending=False)
counts_df

,Token,Count
16,NAME,55826
4,LOCATION,12114
19,EMAIL,7913
6,JOB,3369
5,MASKEDNUMBER,1467
18,URL,996
3,USERNAME,713
20,IP,541
22,PASSWORD,289
9,GENDER,255


In [25]:
# # Define list of problematic row indices
# list_of_weird_rows = [36484,42606,39778,39591,38894,35859,35398,34377,33200,32786,32633,31525,31171,30957,30419,30129,28097,25492,25335,24531,24497,23920,22670,22152,19842,19644,19461,18718,16604,16297,16239,15847,15775,15549,14604,13642,13103,12358,11358,10344,8652,738,6801,5071,4265,3654,3381,3218,1906,38778,36440,35086,7368,13681,29797,7384,13709,29863,35170,38822,30957]

# # Remove duplicates
# list_of_weird_rows = list(set(list_of_weird_rows))
# print(f"Number of rows in list_of_weird_rows: {len(list_of_weird_rows)}")

# # Check if these rows actually exist in the DataFrame
# existing_indices = set(list_of_weird_rows).intersection(set(aws_prepping.index))
# print(f"Number of weird rows found in DataFrame: {len(existing_indices)}")

# # Original row count
# original_count = len(aws_prepping)
# print(f"Original number of rows: {original_count}")

# # Drop the rows
# aws_prepping = aws_prepping.drop(index=list_of_weird_rows, errors='ignore')

# # Reset index to have sequential row numbers
# aws_prepping = aws_prepping.reset_index(drop=True)
# print(f"Number of rows after dropping weird rows: {len(aws_prepping)}")
# print(f"Rows removed: {original_count - len(aws_prepping)}")

# aws_prepping.head()

# df_full = aws_prepping.copy()


In [26]:
# Find rows containing [BIC] in the Template column
bic_rows = aws_prepping[aws_prepping["Template"].str.contains(r"\[BIC\]", na=False)]

if len(bic_rows) > 0:
	print(f"Found {len(bic_rows)} rows containing BIC:")
	for idx, row in bic_rows.iterrows():
		print(f"\nRow Index: {idx}")
		print(f"Template: {row['Template']}")
		print(f"Filled Template: {row['Filled Template']}")
		print("-" * 80)
else:
	print("No rows containing [BIC] found in the Template column.")

# Check if BIC is in any of the other fields
for idx, row in aws_prepping.iterrows():
	if "BIC" in str(row["Filled Template"]) and "BIC" not in str(row["Template"]):
		print(f"\nFound BIC in Filled Template but not in Template at row {idx}:")
		print(f"Template: {row['Template']}")
		print(f"Filled Template: {row['Filled Template']}")
		print("-" * 80)

No rows containing [BIC] found in the Template column.


In [27]:
# Count occurrences of each cleaned token in df_full
counts = {}
for token in unique_matches:
    # Adjust pattern to count tokens without numeric suffixes (e.g., "[NAME]")
    pattern = r'\[' + token + r'\]'
    counts[token] = int(aws_prepping["Template"].dropna().str.count(pattern).sum())

# Convert to a DataFrame and sort by count for a nicer display
counts_df = pd.DataFrame(list(counts.items()), columns=['Token', 'Count']).sort_values(by='Count', ascending=False)
counts_df

,Token,Count
16,NAME,55826
4,LOCATION,12114
19,EMAIL,7913
6,JOB,3369
5,MASKEDNUMBER,1467
18,URL,996
3,USERNAME,713
20,IP,541
22,PASSWORD,289
9,GENDER,255


In [28]:
aws_prepping["Filled Template"][740]

'Draft a divorce settlement for Jan Dach and Paula Fisher IV, taking into account their joint bank account 11971664.'

In [29]:
import re
import pandas as pd

def transform_dataframe(df, doc_name="aws_prepping.csv"):
    # This regex matches placeholders like [NAME], [EMAIL], [ANYTHING_123], etc.
    placeholder_pattern = re.compile(r'(\[[^\]]+\])')
    annotation_rows = []

    for i, row in df.iterrows():
        template_text = str(row["Template"])
        filled_text   = str(row["Filled Template"])
        
        # Split template into chunks + placeholders, e.g.:
        # ["Could you draft a letter for ", "[NAME]", " to send to ", "[EMAIL]", ...]
        parts = placeholder_pattern.split(template_text)
        
        # We'll keep track of where we are in the filled_text
        filled_pos = 0
        
        if i == 741:
            print("Row 741 details:")
            print(row.to_dict())
            print("Parts:", parts)
        if i == 740:
            print("Row 740 details:")
            print(row.to_dict())
            print("Parts:", parts)


        # Walk through each chunk or placeholder
        for idx, part in enumerate(parts):
            # Check if 'part' is a placeholder (e.g., "[NAME]")
            if placeholder_pattern.match(part):
                # Extract the entity type from the placeholder text, e.g. "[NAME]" -> "NAME"
                entity_type = part.strip("[]")
                
                # Find the next literal chunk in parts (the text that comes after this placeholder)
                next_chunk = ""
                for j in range(idx + 1, len(parts)):
                    if not placeholder_pattern.match(parts[j]):
                        next_chunk = parts[j]
                        break
                
                # If there's no next chunk, the replaced text goes until the end of filled_text
                if next_chunk == "":
                    replaced_text = filled_text[filled_pos:]
                    begin_offset = filled_pos
                    end_offset   = filled_pos + len(replaced_text)
                    
                    if replaced_text.strip():
                        annotation_rows.append([
                            doc_name,   # File
                            i,          # Line (row index)
                            begin_offset,
                            end_offset,
                            entity_type
                        ])
                    
                    filled_pos = end_offset
                
                else:
                    # Instead of simply using find(), check if the filled text ends with the next chunk.
                    # This avoids capturing an occurrence of next_chunk that is embedded within the replaced text.
                    if filled_text.endswith(next_chunk):
                        next_chunk_pos = filled_text.rfind(next_chunk, filled_pos)
                    else:
                        next_chunk_pos = filled_text.find(next_chunk, filled_pos)
                    
                    if next_chunk_pos == -1:
                        # If not found, assume the replaced text is everything to the end
                        replaced_text = filled_text[filled_pos:]
                        begin_offset  = filled_pos
                        end_offset    = filled_pos + len(replaced_text)
                        
                        if replaced_text.strip():
                            annotation_rows.append([
                                doc_name,
                                i,
                                begin_offset,
                                end_offset,
                                entity_type
                            ])
                        filled_pos = end_offset
                    else:
                        # The replaced text is everything from filled_pos up to where next_chunk starts
                        replaced_text = filled_text[filled_pos:next_chunk_pos]
                        begin_offset  = filled_pos
                        end_offset    = next_chunk_pos
                        
                        if replaced_text.strip():
                            annotation_rows.append([
                                doc_name,
                                i,
                                begin_offset,
                                end_offset,
                                entity_type
                            ])
                        
                        filled_pos = next_chunk_pos
            
            else:
                # This is a normal text chunk (not a placeholder).
                pos = filled_text.find(part, filled_pos)
                if pos != -1:
                    filled_pos = pos + len(part)
                else:
                    pass

    ann_df = pd.DataFrame(annotation_rows, columns=["File", "Line", "Begin Offset", "End Offset", "Type"])
    
    return ann_df

# Transform the DataFrame and save the resulting CSV
annotations_df = transform_dataframe(aws_prepping)
print(annotations_df.head())
print("Unique types:", annotations_df["Type"].unique())
print(len(annotations_df["Type"].unique()))
print(len(annotations_df))
print(annotations_df["Type"].value_counts())


Row 740 details:
{'Template': 'Draft a divorce settlement for [NAME] [NAME] and [NAME] [NAME], taking into account their joint bank account [MASKEDNUMBER].', 'Filled Template': 'Draft a divorce settlement for Jan Dach and Paula Fisher IV, taking into account their joint bank account 11971664.'}
Parts: ['Draft a divorce settlement for ', '[NAME]', ' ', '[NAME]', ' and ', '[NAME]', ' ', '[NAME]', ', taking into account their joint bank account ', '[MASKEDNUMBER]', '.']
Row 741 details:
{'Template': "Hey, I've been struggling with stress lately, can you send me a mindfulness-based therapy guide to my email at [EMAIL]?", 'Filled Template': "Hey, I've been struggling with stress lately, can you send me a mindfulness-based therapy guide to my email at Nicklaus_Erdman91@hotmail.com?"}
Parts: ["Hey, I've been struggling with stress lately, can you send me a mindfulness-based therapy guide to my email at ", '[EMAIL]', '?']
               File  Line  Begin Offset  End Offset   Type
0  aws_preppi

In [30]:
aws_prepping["Filled Template"][740]

'Draft a divorce settlement for Jan Dach and Paula Fisher IV, taking into account their joint bank account 11971664.'

In [31]:
counts = {}
for token in unique_matches:
    # Adjust pattern to count tokens without numeric suffixes (e.g., "[NAME]")
    pattern = r'\[' + token + r'\]'
    counts[token] = int(aws_prepping["Template"].dropna().str.count(pattern).sum())

# Convert to a DataFrame and sort by count for a nicer display
counts_df = pd.DataFrame(list(counts.items()), columns=['Token', 'Count']).sort_values(by='Count', ascending=False)
counts_df

,Token,Count
16,NAME,55826
4,LOCATION,12114
19,EMAIL,7913
6,JOB,3369
5,MASKEDNUMBER,1467
18,URL,996
3,USERNAME,713
20,IP,541
22,PASSWORD,289
9,GENDER,255


In [32]:
aws_prepping["Filled Template"][740]

'Draft a divorce settlement for Jan Dach and Paula Fisher IV, taking into account their joint bank account 11971664.'

In [33]:
def mask_text_using_annotations(text, anns):
    # Sort annotations in reverse order of Begin Offset to avoid shifting issues.
    for ann in sorted(anns, key=lambda x: x["Begin Offset"], reverse=True):
        start = ann["Begin Offset"]
        end = ann["End Offset"]
        
        # Check for valid offset ranges.
        if start < 0 or end > len(text):
            print(f"Warning: Annotation offsets {start}-{end} are out of bounds for text length {len(text)}")
            continue

        # Replace the span with the masked placeholder.
        text = text[:start] + f"[{ann['Type']}]" + text[end:]
    
    return text

def apply_masking(row):
    # Retrieve all annotations corresponding to the current row.
    # Subtract 2 from the row index to match the Line number in annotations_df
    line_num = row.name
    anns = annotations_df[annotations_df["Line"] == line_num].to_dict(orient="records")
    if anns:
        return mask_text_using_annotations(row["Filled Template"], anns)
    return row["Filled Template"]

tqdm.pandas(desc="Applying masking")
aws_prepping["Masked Filled Template"] = aws_prepping.progress_apply(apply_masking, axis=1)


Applying masking: 100%|██████████| 42699/42699 [00:22<00:00, 1938.43it/s]


In [34]:
aws_prepping["Filled Template"][740]

'Draft a divorce settlement for Jan Dach and Paula Fisher IV, taking into account their joint bank account 11971664.'

In [35]:
annotations_df[annotations_df["Line"] == id]

,File,Line,Begin Offset,End Offset,Type
76405,aws_prepping.csv,38860,99,124,URL


In [36]:
annotations_df

,File,Line,Begin Offset,End Offset,Type
0,aws_prepping.csv,0,0,5,NAME
1,aws_prepping.csv,1,18,44,NAME
2,aws_prepping.csv,2,58,64,NAME
3,aws_prepping.csv,2,65,71,NAME
4,aws_prepping.csv,3,53,76,EMAIL
...,...,...,...,...,...
83829,aws_prepping.csv,42696,71,82,NAME
83830,aws_prepping.csv,42697,56,72,NAME
83831,aws_prepping.csv,42697,95,111,LOCATION
83832,aws_prepping.csv,42698,76,82,NAME


In [37]:
aws_prepping

,Template,Filled Template,Masked Filled Template
0,[NAME] needs help designing a user-friendly ch...,Halie needs help designing a user-friendly che...,[NAME] needs help designing a user-friendly ch...
1,Draft a letter to [NAME] explaining the import...,"Draft a letter to Runte, McGlynn and Kautzer e...",Draft a letter to [NAME] explaining the import...
2,3. Describe the process of conducting an inter...,3. Describe the process of conducting an inter...,3. Describe the process of conducting an inter...
3,Can you tell me how companies should handle cu...,Can you tell me how companies should handle cu...,Can you tell me how companies should handle cu...
4,Write a research proposal for [NAME] to study ...,Write a research proposal for Wolf - Reilly to...,Write a research proposal for [NAME] to study ...
...,...,...,...
42694,Can you write a report on the recent developme...,Can you write a report on the recent developme...,Can you write a report on the recent developme...
42695,Can you compile a list of top 10 competitors i...,Can you compile a list of top 10 competitors i...,Can you compile a list of top 10 competitors i...
42696,Please provide a list of the top 5 corporate g...,Please provide a list of the top 5 corporate g...,Please provide a list of the top 5 corporate g...
42697,Can you help me with a workplace culture asses...,Can you help me with a workplace culture asses...,Can you help me with a workplace culture asses...


In [38]:
id = 3218

In [39]:
aws_prepping["Masked Filled Template"][id]

'Can you write a guide on the role of a family therapist for [NAME]?'

In [40]:
aws_prepping["Template"][id]

'Can you write a guide on the role of a family therapist for [NAME]?'

In [41]:
aws_prepping["Filled Template"][id]

'Can you write a guide on the role of a family therapist for McKenzie LLC?'

In [42]:
aws_prepping.head()

,Template,Filled Template,Masked Filled Template
0,[NAME] needs help designing a user-friendly ch...,Halie needs help designing a user-friendly che...,[NAME] needs help designing a user-friendly ch...
1,Draft a letter to [NAME] explaining the import...,"Draft a letter to Runte, McGlynn and Kautzer e...",Draft a letter to [NAME] explaining the import...
2,3. Describe the process of conducting an inter...,3. Describe the process of conducting an inter...,3. Describe the process of conducting an inter...
3,Can you tell me how companies should handle cu...,Can you tell me how companies should handle cu...,Can you tell me how companies should handle cu...
4,Write a research proposal for [NAME] to study ...,Write a research proposal for Wolf - Reilly to...,Write a research proposal for [NAME] to study ...


In [43]:
# Compare the two strings
counter = 0
for i, row in aws_prepping.iterrows():
    masked = aws_prepping["Masked Filled Template"][i]
    template = aws_prepping["Template"][i]

    # Check if they are equal
    if masked != template:
        print("="*80)
        print(f"Masked Filled Template at index {i}:")
        print(masked)
        print(f"\nTemplate at index {i}:")
        print(template)
        print(f"\nOrginal Filled Template at index {i}:")
        print(aws_prepping["Filled Template"][i])
        print("\nThey are different.")
        counter += 1

print(f"errors in masking: {counter}")
        

errors in masking: 0


In [44]:
# Create a new dataframe with just the 'Filled Template' column
aws_prepping = pd.DataFrame(aws_prepping['Filled Template'])

# Display the first few rows of the new dataframe
print(aws_prepping.head())



                                     Filled Template
0  Halie needs help designing a user-friendly che...
1  Draft a letter to Runte, McGlynn and Kautzer e...
2  3. Describe the process of conducting an inter...
3  Can you tell me how companies should handle cu...
4  Write a research proposal for Wolf - Reilly to...


In [45]:
aws_prepping

,Filled Template
0,Halie needs help designing a user-friendly che...
1,"Draft a letter to Runte, McGlynn and Kautzer e..."
2,3. Describe the process of conducting an inter...
3,Can you tell me how companies should handle cu...
4,Write a research proposal for Wolf - Reilly to...
...,...
42694,Can you write a report on the recent developme...
42695,Can you compile a list of top 10 competitors i...
42696,Please provide a list of the top 5 corporate g...
42697,Can you help me with a workplace culture asses...


In [46]:
aws_prepping["Filled Template"][740]

'Draft a divorce settlement for Jan Dach and Paula Fisher IV, taking into account their joint bank account 11971664.'

In [ ]:
aws_prepping.to_csv("aws_prepping.csv",index=False, header=False)
annotations_df.to_csv("aws_ann_data.csv",index=False)
# Split data into training (first 90%) and testing (last 10%)
total_rows = len(aws_prepping)
split_idx = int(total_rows * 0.9)
training_data = aws_prepping.iloc[:split_idx]
testing_data = aws_prepping.drop(training_data.index)
ann_traning_data = annotations_df[annotations_df["Line"].isin(training_data.index)]
ann_testing_data = annotations_df[annotations_df["Line"].isin(testing_data.index)]



training_data.to_csv("aws_training_data.csv",index=False, header=False)
testing_data.to_csv("aws_testing_data.csv",index=False, header=False)

ann_traning_data.to_csv("aws_ann_training_data.csv",index=False)
ann_testing_data.to_csv("aws_ann_testing_data.csv",index=False)

In [48]:
# First and last line in training data
print("Training Data:")
print(f"First row (index {training_data.index[0]}):")
print(training_data.iloc[0]['Filled Template'])
print("\nLast row (index {training_data.index[-1]}):")
print(training_data.iloc[-1]['Filled Template'])

# First and last line in testing data
print("\nTesting Data:")
print(f"First row (index {testing_data.index[0]}):")
print(testing_data.iloc[0]['Filled Template']) 
print("\nLast row (index {testing_data.index[-1]}):")
print(testing_data.iloc[-1]['Filled Template'])

# Summary statistics
print("\nSummary:")
print(f"Training data shape: {training_data.shape}")
print(f"Testing data shape: {testing_data.shape}")
print(f"Percentage of data in testing set: {testing_data.shape[0]/(training_data.shape[0]+testing_data.shape[0]):.2%}")



Training Data:
First row (index 0):
Halie needs help designing a user-friendly checkout process for their online store.

Last row (index {training_data.index[-1]}):
We should prepare a report on our current compliance status and share it with Vivienne_Quitzon34@hotmail.com to ensure we're on the right track.

Testing Data:
First row (index 38429):
Write a review of the latest neuropsychology textbook for Wolff Group to share with their students at Marielle_Herzog@yahoo.com.

Last row (index {testing_data.index[-1]}):
During our video conference, can you explain how cognitive therapy can help Robert Farrell manage their anger issues?

Summary:
Training data shape: (38429, 1)
Testing data shape: (4270, 1)
Percentage of data in testing set: 10.00%


## AWS Comprehend dataprepping END

In [49]:
from IPython.display import clear_output

for entities in unique_matches:
	regex_pattern = r'\[' + entities + r'\]'
	sextype_examples = df_full[df_full['Template'].str.contains(regex_pattern, na=False)]

	if sextype_examples.empty:
		print("No rows found with the token [SEXTYPE]")
	else:
		num_rows = min(20, len(sextype_examples))
		for i in range(num_rows):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)
	#input("Press Enter to load next value...")
	#clear_output(wait=True)
	

No rows found with the token [SEXTYPE]
No rows found with the token [SEXTYPE]

Please write a research paper on the legal implications of tracking user [IP], [IP], and [USERAGENT] data without consent.
Please write a research paper on the legal implications of tracking user 1751:a95b:05b9:ce8b:431e:ecfe:ce30:bbdd, 21:f7:ae:7f:94:f6, and Mozilla/5.0 (Windows NT 5.0; WOW64; rv:14.3) Gecko/20100101 Firefox/14.3.0 data without consent.


I'm interested in learning more about the privacy implications of collecting [USERAGENT] data. Can you help me out?
I'm interested in learning more about the privacy implications of collecting Opera/11.35 (X11; Linux x86_64; U; VO Presto/2.9.176 Version/12.00) data. Can you help me out?


Could you write a press release about the new ADHD therapy center opening at [LOCATION] [USERAGENT] Street?
Could you write a press release about the new ADHD therapy center opening at 2243 Mozilla/5.0 (Macintosh; U; Intel Mac OS X 10_8_9)  AppleWebKit/537.1.0 (KHTML, lik

# Short form removal

In [50]:
# contraction expansion

def remove_short_form(text):
    if isinstance(text, str):
        contractions = {
            # Common negatives
            "isn't": "is not",
            "wasn't": "was not",
            "weren't": "were not",
            "haven't": "have not",
            "hasn't": "has not",
            "hadn't": "had not",
            "don't": "do not",
            "doesn't": "does not",
            "didn't": "did not",
            "won't": "will not",
            "wouldn't": "would not",
            "shouldn't": "should not",
            "couldn't": "could not",
            "mustn't": "must not",
            "shan't": "shall not",
            
            # Pronoun contractions
            "i'm": "i am",
            "you're": "you are",
            "he's": "he is",
            "she's": "she is",
            "it's": "it is",
            "we're": "we are",
            "they're": "they are",

            # Possession / descriptive
            "who's": "who is",
            "what's": "what is",
            "where's": "where is",
            "when's": "when is",
            "why's": "why is",
            "how's": "how is",
            "there's": "there is",
            "here's": "here is",
            "that's": "that is",

            # Past and hypothetical
            "i'd": "i would",
            "you'd": "you would",
            "he'd": "he would",
            "she'd": "she would",
            "we'd": "we would",
            "they'd": "they would",

            # Future forms
            "i'll": "i will",
            "you'll": "you will",
            "he'll": "he will",
            "she'll": "she will",
            "it'll": "it will",
            "we'll": "we will",
            "they'll": "they will",

            # Perfect tense
            "i've": "i have",
            "you've": "you have",
            "we've": "we have",
            "they've": "they have",
            "he's": "he has",
            "she's": "she has",
            "it's": "it has",

            # Imperative/other
            "let's": "let us",
            "y'all": "you all",
            "o'clock": "of the clock",
            "ma'am": "madam",
            "gonna": "going to",
            "wanna": "want to",
            "gotta": "got to",
            "ain't": "is not"
        }

        # Replace contractions case-insensitively
        for contraction, replacement in contractions.items():
            text = re.sub(rf"\b{re.escape(contraction)}\b", replacement, text, flags=re.IGNORECASE)

        return text
    return text



# Normalization

In [51]:
def normalize_text(text):
	# text to lower
	parts = re.split(r'(\[[^\]]*\])', text)

	processed_parts = []
	for segment in parts:
		if segment.startswith('[') and segment.endswith(']'):
			processed_parts.append(segment)
		else:
			processed_parts.append(segment.lower())

	text = "".join(processed_parts)
	# remove extra spaces
	text = re.sub(r'\s+', ' ', text)
	# Use the function to remove short forms
	text = remove_short_form(text)
	return text

In [52]:
from tqdm import tqdm

tqdm.pandas(desc="Normalizing Template")
df_full['Template'] = df_full['Template'].progress_apply(normalize_text)
tqdm.pandas(desc="Normalizing Filled Template")
df_full['Filled Template'] = df_full['Filled Template'].progress_apply(normalize_text)

Normalizing Filled Template: 100%|██████████| 42699/42699 [00:07<00:00, 5383.98it/s]


# Lemming

In [53]:

# Load the English NLP model
nlp = spacy.load("en_core_web_sm")# TODO: Make this use the large model
spacy.prefer_gpu()

def lemmatize_text(text):
    # Check for missing values
    if pd.isna(text):
        return text
    # Process the text with spaCy
    doc = nlp(text)
    # Join lemmatized tokens back into a string
    return " ".join(token.lemma_ for token in doc)

In [54]:

try:
	# Try loading the lemmatized data
	df_full = pd.read_csv("lemmatized_data.csv")
	print("Loaded data from lemmatized_data.csv")
except FileNotFoundError:
	# If the file doesn't exist, perform lemmatization
	print("lemmatized_data.csv not found, performing lemmatization...")
	tqdm.pandas(desc="Lemmatizing Template")
	df_full['Template'] = df_full['Template'].progress_apply(lemmatize_text)
	tqdm.pandas(desc="Lemmatizing Filled Template")
	df_full['Filled Template'] = df_full['Filled Template'].progress_apply(lemmatize_text)

	# Check the results
	print(df_full.head())
	#save
	df_full.to_csv("lemmatized_data.csv", index=False)

Loaded data from lemmatized_data.csv


In [55]:
# Count the labeled items in the DataFrame
total_labeled_items = count_labeled_items(df_full)
print(f"Total number of labeled items: {total_labeled_items}")

Total number of labeled items: 83834


# Sentences Splitting

In [56]:
def split_into_sentences(text):
    """
    Splits a given text (string) into a list of sentences using spaCy's sentence segmentation.
    Returns an empty list if the input is None or NaN.
    """
    if pd.isna(text):
        return []
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

In [57]:
import os

if not os.path.exists("sentences_data.csv"):
	tqdm.pandas(desc="Splitting Sentences")
	
	df_full["Template_Sentences"] = df_full["Template"].progress_apply(split_into_sentences)
	df_full["Filled_Sentences"]   = df_full["Filled Template"].progress_apply(split_into_sentences)
	
	# Optionally, save the updated dataframe
	df_full.to_csv("sentence_data.csv", index=False)
	
	print("Sentence splitting complete. Data saved to sentence_data.csv")
	print(df_full.head())
else:
	print("sentence_data.csv already exists. Skipping sentence splitting.")


Splitting Sentences: 100%|██████████| 42699/42699 [06:21<00:00, 111.86it/s]


Sentence splitting complete. Data saved to sentence_data.csv
                                            Template  \
0  in our video conference , discuss the role of ...   
1  could you draft a letter for [ name ] to send ...   
2  discuss the option for [ name ] [ name ] who w...   
3  13 . write a press release announce [ name ] [...   
4  9 . develop an inventory management plan for [...   

                                     Filled Template  \
0  in our video conference , discuss the role of ...   
1  could you draft a letter for dietrich , schuli...   
2  discuss the option for jeffery pfeffer who wan...   
3  13 . write a press release announce gayle wate...   
4  9 . develop an inventory management plan for e...   

                                  Template_Sentences  \
0  [in our video conference , discuss the role of...   
1  [could you draft a letter for [ name ] to send...   
2  [discuss the option for [ name ] [ name ] who ...   
3  [13 ., write a press release announce 

## VIEW DATA

In [58]:
df_full.head()

,Template,Filled Template,Template_Sentences,Filled_Sentences
0,"in our video conference , discuss the role of ...","in our video conference , discuss the role of ...","[in our video conference , discuss the role of...","[in our video conference , discuss the role of..."
1,could you draft a letter for [ name ] to send ...,"could you draft a letter for dietrich , schuli...",[could you draft a letter for [ name ] to send...,"[could you draft a letter for dietrich , schul..."
2,discuss the option for [ name ] [ name ] who w...,discuss the option for jeffery pfeffer who wan...,[discuss the option for [ name ] [ name ] who ...,[discuss the option for jeffery pfeffer who wa...
3,13 . write a press release announce [ name ] [...,13 . write a press release announce gayle wate...,"[13 ., write a press release announce [ name ]...","[13 ., write a press release announce gayle wa..."
4,9 . develop an inventory management plan for [...,9 . develop an inventory management plan for e...,[9 . develop an inventory management plan for ...,[9 . develop an inventory management plan for ...


In [59]:
from IPython.display import clear_output

for entities in unique_matches:
	print(entities)

	entities = entities.lower()
	print(entities)
	regex_pattern = r'\[ ' + entities + r' \]'
	sextype_examples = df_full[df_full['Template'].str.contains(regex_pattern, na=False)]

	if sextype_examples.empty:
		print("No rows found with the token")
	else:
		num_rows = min(20, len(sextype_examples))
		for i in range(num_rows):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)
	#input("Press Enter to load next value...")
	#clear_output(wait=True)
	

ETHEREUMADDRESS
ethereumaddress
No rows found with the token
LITECOINADDRESS
litecoinaddress
No rows found with the token
USERAGENT
useragent

12 . monitor the website 's loading speed and optimize it by analyze datum from [ useragent ] and [ IP ] .
12 . monitor the website 's loading speed and optimize it by analyze datum from mozilla/5.0 ( macintosh ; ppc mac os x 10_6_9 rv:3.0 ; hi ) applewebkit/536.0.0 ( khtml , like gecko ) version/4.0.6 safari/536.0.0 and fb65 : fec4 : bc09:1108 : cb0d : ff05:9da6 : cfa7 .


how can I ensure that I be follow privacy law when deal with [ URL ] and [ useragent ] information ?
how can I ensure that I be follow privacy law when deal with https://dry-steam.com/ and mozilla/5.0 ( compatible ; msie 7.0 ; window not 5.2 ; trident/5.0 ; .net clr 1.7.22860.9 ) information ?


I be interested in learn more about the privacy implication of collect [ useragent ] datum . can you help I out ?
I be interested in learn more about the privacy implication of collec

In [60]:
# Define a dataset class to store our data splits
class dataset:
    def __init__(self, name, dataset, training_data=None, testing_data=None):
        self.name = name
        self.dataset = dataset
        self.training_data = training_data
        self.testing_data = testing_data
        self.text_accuracy = 0
        self.precision = 0
        self.recall = 0
        self.f1 = 0
        self.nlp = None
    
    def __str__(self):
        return f"Dataset {self.name}: {len(self.dataset)} total rows, {len(self.training_data)} training, {len(self.testing_data)} testing"

# Create a single dataset with 100% of the data
test_pct = 1  # 100% of the data

split_index = int(len(df_full) * 0.9)

# Split the dataset into training (90%) and testing (10%)
training_data = df_full[:split_index]
testing_data = df_full[split_index:]

# Create a dataset instance with the splits
result = dataset(
    name=test_pct, 
    dataset=df_full, 
    training_data=training_data, 
    testing_data=testing_data
)

# Store in datasets dictionary if needed
datasets = {test_pct: result}

print(f"Total dataset size: {len(df_full)}")
print(f"Training set size: {len(training_data)} ({len(training_data)/len(df_full):.1%})")
print(f"Test set size: {len(testing_data)} ({len(testing_data)/len(df_full):.1%})")

# Save the datasets to disk
training_data.to_csv("training_data.csv", index=False)
testing_data.to_csv("test_data.csv", index=False)

Total dataset size: 42699
Training set size: 38429 (90.0%)
Test set size: 4270 (10.0%)


### NLP Implementation 

In [61]:
analyzer = AnalyzerEngine()

In [62]:
# test Call analyzer to get results
results = analyzer.analyze(text="My phone number is 212-555-5555",
                           entities=["PHONE_NUMBER"],
                           language='en')
print(results)

[type: PHONE_NUMBER, start: 19, end: 31, score: 0.75]


In [63]:
def clean_text(text):
    """Clean and normalize text."""
    return str(text).strip()

def extract_entities(template, filled):
    """
    Extract entities by aligning placeholders (e.g. "[NAME_1]" => "NAME")
    with the actual text that replaced them in `filled`.
    """
    entities = []
    i = 0  # pointer into template
    j = 0  # pointer into filled

    while i < len(template) and j < len(filled):
        if template[i] == '[':
            # Found a placeholder
            closing = template.find(']', i)
            if closing == -1:
                break  # malformed placeholder; exit
            placeholder = template[i+1:closing]  # e.g. "NAME_1"
            label = re.sub(r'_\d+', '', placeholder)  # becomes "NAME"

            next_i = closing + 1
            # Look for next bracket to capture literal text that follows the placeholder
            next_bracket = template.find('[', next_i)
            literal = template[next_i:] if next_bracket == -1 else template[next_i:next_bracket]

            if literal:
                literal_index = filled.find(literal, j)
                if literal_index == -1:
                    # If literal is missing, log a warning and skip this placeholder
                    print(f"WARNING: Could not find literal '{literal}' after placeholder '{label}'. Skipping.")
                    i = closing + 1
                    continue
                else:
                    entity_start = j
                    entity_end = literal_index
                    j = literal_index  # advance pointer j to the literal start
            else:
                # No literal after the placeholder; log info and skip
                print(f"INFO: No literal after placeholder '[{placeholder}]'. Skipping.")
                i = closing + 1
                continue

            entities.append((entity_start, entity_end, label))
            i = closing + 1
        else:
            if template[i] == filled[j]:
                i += 1
                j += 1
            else:
                j += 1

    return entities

In [ ]:
def NLP_training(df_training, df_testing):
    """
    Train an NER model using df_training and evaluate using df_testing.
    Returns text accuracy, label-level precision, recall, F1 score, 
    detailed error counts, and the trained nlp model.
    """
    import random
    import spacy
    from spacy.training.example import Example
    from spacy.util import minibatch
    
    # ------------------------------
    # Step 0: Build and Clean Training Data
    # ------------------------------
    raw_train_data = []
    for _, row in df_training.iterrows():
        template = clean_text(row['Template'])
        filled = clean_text(row['Filled Template'])
        entities = extract_entities(template, filled)
        if entities:
            raw_train_data.append((filled, {"entities": entities}))
    
    # ------------------------------
    # Step 0.5: Re-align Entity Offsets to Token Boundaries
    # ------------------------------
    tokenizer_nlp = spacy.blank("en")
    aligned_train_data = []
    for text, annotation in raw_train_data:
        doc = tokenizer_nlp(text)
        new_entities = []
        for start, end, label in annotation["entities"]:
            # Expand to token boundaries.
            span = doc.char_span(start, end, alignment_mode="expand")
            if span is not None:
                new_entities.append((span.start_char, span.end_char, label))
            else:
                print(f"WARNING: Could not align entity '{text[start:end]}' in text: {text}")
        if new_entities:
            aligned_train_data.append((text, {"entities": new_entities}))
    
    TRAIN_DATA = aligned_train_data
    
    # ------------------------------
    # Step 1: Optionally shuffle or further split training data if needed
    # ------------------------------
    # For now, we assume df_training is fully for training, and df_testing will be used for evaluation.
    
    # ------------------------------
    # Step 2: Create and Configure the Model
    # ------------------------------
    nlp = spacy.blank("en")
    if "ner" not in nlp.pipe_names:
        ner = nlp.add_pipe("ner", last=True)
    else:
        ner = nlp.get_pipe("ner")
    
    # Add labels from training data.
    for _, annotations in TRAIN_DATA:
        for _, _, label in annotations["entities"]:
            ner.add_label(label)
    
    # ------------------------------
    # Step 3: Train the Model Using Batches with Dropout
    # ------------------------------
    optimizer = nlp.begin_training()
    n_iter = 20  # Number of epochs
    batch_size = 16
    for itn in range(n_iter):
        random.shuffle(TRAIN_DATA)
        batches = minibatch(TRAIN_DATA, size=batch_size)
        losses = {}
        for batch in batches:
            examples = []
            for text, annotations in batch:
                doc = nlp.make_doc(text)
                examples.append(Example.from_dict(doc, annotations))
            nlp.update(examples, sgd=optimizer, drop=0.3, losses=losses)
        print(f"Iteration {itn + 1}/{n_iter} - Losses: {losses}")
    
    # ------------------------------
    # Step 4: Evaluate on df_testing with Detailed Metrics
    # ------------------------------
    total_texts = 0
    text_correct_count = 0
    TP = 0  # true positives for individual entity predictions
    FP = 0  # false positives: entities predicted but not in ground truth
    FN = 0  # false negatives: ground truth entities not predicted
    failed_entities = []  # collect details on mismatches
    
    for _, row in df_testing.iterrows():
        text = clean_text(row['Filled Template'])
        # Extract ground truth entities from the provided template/fill pair.
        template = clean_text(row['Template'])
        ground_entities = extract_entities(template, text)
        # Align ground truth entities to token boundaries.
        doc = nlp.make_doc(text)
        aligned_ground = []
        for start, end, label in ground_entities:
            span = doc.char_span(start, end, alignment_mode="expand")
            if span is not None:
                aligned_ground.append((span.start_char, span.end_char, label))
            else:
                print(f"WARNING: Could not align ground truth entity '{text[start:end]}'")
        # Get predicted entities from the model.
        pred_doc = nlp(text)
        predicted = [(ent.start_char, ent.end_char, ent.label_) for ent in pred_doc.ents]
        
        # Text-level evaluation: exact match of entity sets.
        if set(predicted) == set(aligned_ground):
            text_correct_count += 1
        
        # Label-level evaluation.
        for entity in aligned_ground:
            total_texts += 0  # placeholder if needed; not used in label-level metrics
            if entity in predicted:
                TP += 1
            else:
                FN += 1
                failed_entities.append({"expected": entity, "text": text})
        for entity in predicted:
            if entity not in aligned_ground:
                FP += 1
        
        total_texts += 1
    
    # Compute text-level accuracy.
    text_accuracy = text_correct_count / total_texts if total_texts > 0 else 0
    
    # Compute precision, recall and F1 for entity-level predictions.
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    print("\n=== Evaluation on Testing Data ===")
    print(f"Text Accuracy (exact match): {text_accuracy:.2%}")
    print(f"Entity-level Precision: {precision:.2%}")
    print(f"Entity-level Recall: {recall:.2%}")
    print(f"Entity-level F1 Score: {f1:.2%}")
    print(f"False Positives: {FP}")
    print(f"False Negatives: {FN}")
    
    return text_accuracy, precision, recall, f1, {"FP": FP, "FN": FN, "failed_entities": failed_entities}, nlp

def mask_pii(text, model):
        """
        Mask detected entities in the text with their label names.
        Entities are replaced from the end of the text to avoid offset issues.
        """
        doc = model(text)
        spans = [(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents]
        spans = sorted(spans, key=lambda x: x[0], reverse=True)
        masked_text = text
        for start, end, label in spans:
            masked_text = masked_text[:start] + f"[{label}]" + masked_text[end:]
        return masked_text


In [65]:
for key, ds_obj in datasets.items():
    print(f"Processing dataset with test percentage: {ds_obj.name}")
    # ds_obj.dataset_training and ds_obj.dataset_testing should be defined in each dataset object.
    text_acc, precision, recall, f1, error_details, nlp_model = NLP_training(ds_obj.training_data, ds_obj.testing_data)
    ds_obj.text_accuracy = text_acc
    ds_obj.precision = precision
    ds_obj.recall = recall
    ds_obj.f1 = f1
    ds_obj.nlp = nlp_model
    print(ds_obj)
    

Processing dataset with test percentage: 1
INFO: No literal after placeholder '[ url ]'. Skipping.
INFO: No literal after placeholder '[ url ]'. Skipping.
INFO: No literal after placeholder '[ url ]'. Skipping.
INFO: No literal after placeholder '[ name ]'. Skipping.
INFO: No literal after placeholder '[ name ]'. Skipping.
INFO: No literal after placeholder '[ name ]'. Skipping.
INFO: No literal after placeholder '[ name ]'. Skipping.
INFO: No literal after placeholder '[ location ]'. Skipping.
INFO: No literal after placeholder '[ url ]'. Skipping.
INFO: No literal after placeholder '[ url ]'. Skipping.
INFO: No literal after placeholder '[ name ]'. Skipping.
INFO: No literal after placeholder '[ name ]'. Skipping.
INFO: No literal after placeholder '[ name ]'. Skipping.
INFO: No literal after placeholder '[ name ]'. Skipping.
INFO: No literal after placeholder '[ name ]'. Skipping.
INFO: No literal after placeholder '[ name ]'. Skipping.
INFO: No literal after placeholder '[ url ]'. 

In [66]:
for key, ds_obj in datasets.items():
    # Create a copy of the test dataset and add new columns.
    ds = ds_obj.testing_data.copy()
    ds["original_text"] = ds["Filled Template"]
    ds["labeled_text"] = ds["Template"]
    ds["masked_text"] = ds["Filled Template"].apply(lambda x: mask_pii(x, ds_obj.nlp))
    
    # Save the new dataset to CSV
    ds_to_save = ds[["original_text", "labeled_text", "masked_text"]]
    output_filename = f"dataset_{ds_obj.name}.csv"
    ds_to_save.to_csv(output_filename, index=False)
    print(f"Saved {output_filename}")


Saved dataset_1.csv


# EVAL

In [69]:
import pandas as pd
import re
from difflib import SequenceMatcher

# Define the set of known placeholders.
PLACEHOLDERS = {"NAME", "LOCATION", "EMAIL", "JOB", "MASKEDNUMBER",
                "URL", "USERNAME", "IP", "PASSWORD", "GENDER",
                "CREDITCARDNUMBER", "USERAGENT", "IBAN", "BIC"}

def extract_masked_tokens(text, return_positions=False):
    """
    Extracts all masked tokens that match known placeholders.
    If return_positions is True, returns a list of tuples (token, start, end).
    Otherwise, returns just the list of token strings.
    """
    pattern = re.compile(r'\b(' + '|'.join(PLACEHOLDERS) + r')\b', re.IGNORECASE)
    if return_positions:
        return [(m.group(), m.start(), m.end()) for m in pattern.finditer(text)]
    else:
        return pattern.findall(text)

def map_gold_span_to_pred(gold, pred, g_start, g_end):
    """
    Maps a span (g_start, g_end) from the gold text to the corresponding span in the predicted text
    using SequenceMatcher to align the two strings.
    Returns a tuple (pred_start, pred_end). If mapping fails, returns (None, None).
    """
    sm = SequenceMatcher(None, gold, pred)
    opcodes = sm.get_opcodes()
    pred_start = None
    pred_end = None
    for tag, i1, i2, j1, j2 in opcodes:
        # Map the start of the gold token.
        if g_start >= i1 and g_start < i2:
            if tag in ("equal", "replace"):
                pred_start = j1 + (g_start - i1)
            else:
                pred_start = j1
        # Map the end of the gold token.
        if g_end > i1 and g_end <= i2:
            if tag in ("equal", "replace"):
                pred_end = j1 + (g_end - i1)
            else:
                pred_end = j1
        if pred_start is not None and pred_end is not None:
            break
    return pred_start, pred_end

def analyze_row_aligned(original, gold, pred):
    """
    For each gold token, maps its span to the predicted text and then:
      1. Checks if a valid label (one of PLACEHOLDERS) is present at that location.
      2. If a valid label is found, compares it with the gold token:
           - If they match (ignoring case), counts as correct.
           - If they differ, counts as a misplaced token.
      3. If no valid label is found at the mapped span or mapping fails, counts as missing.
    Additionally, any extra valid tokens in the predicted text that do not correspond to any gold token
    are marked as false positives.
    
    Returns:
      - metrics: dictionary with counts for correct, misplaced, missing, false_positive.
      - examples: dictionary of examples for error cases.
    """
    gold_tokens = extract_masked_tokens(gold, return_positions=True)
    metrics = {"correct": 0, "misplaced": 0, "missing": 0, "false_positive": 0}
    examples = {"misplaced": []}
    
    # For each gold token, map its span to the predicted text.
    for idx, (g_token, g_start, g_end) in enumerate(gold_tokens):
        p_start, p_end = map_gold_span_to_pred(gold, pred, g_start, g_end)
        if p_start is None or p_end is None or p_start < 0 or p_end > len(pred):
            metrics["missing"] += 1
            examples["misplaced"].append({
                "gold": g_token,
                "predicted": None,
                "gold_span": (g_start, g_end),
                "pred_span": None,
                "note": "Mapping not found"
            })
        else:
            # Extract the predicted token and strip extra spaces.
            pred_token = pred[p_start:p_end].strip()
            valid_labels = {p.lower() for p in PLACEHOLDERS}
            if pred_token.lower() not in valid_labels:
                metrics["missing"] += 1
                examples["misplaced"].append({
                    "gold": g_token,
                    "predicted": pred_token,
                    "gold_span": (g_start, g_end),
                    "pred_span": (p_start, p_end),
                    "note": "No valid label found at mapped span"
                })
            else:
                if pred_token.lower() == g_token.lower():
                    metrics["correct"] += 1
                else:
                    metrics["misplaced"] += 1
                    examples["misplaced"].append({
                        "gold": g_token,
                        "predicted": pred_token,
                        "gold_span": (g_start, g_end),
                        "pred_span": (p_start, p_end),
                        "note": "Wrong label"
                    })
    
    # Identify extra predicted tokens not mapped from any gold token.
    pred_tokens = extract_masked_tokens(pred, return_positions=True)
    mapped_spans = []
    for (g_token, g_start, g_end) in gold_tokens:
        p_span = map_gold_span_to_pred(gold, pred, g_start, g_end)
        if p_span[0] is not None and p_span[1] is not None:
            mapped_spans.append(p_span)
    for p_token, p_start, p_end in pred_tokens:
        if not any(p_start == m[0] and p_end == m[1] for m in mapped_spans):
            metrics["false_positive"] += 1
            examples["misplaced"].append({
                "gold": None,
                "predicted": p_token,
                "gold_span": None,
                "pred_span": (p_start, p_end),
                "note": "Extra predicted token"
            })
            
    return metrics, examples

def analyze_masking(file_path, original_col=None, gold_col=None, pred_col=None):
    """
    Reads a file (CSV or pickle) that contains your evaluation results. The file is assumed to contain a pandas
    DataFrame with the columns for:
      - Original Text (the unmasked filled text),
      - Labeled Text (the gold masked template), and
      - Masked Text (the predicted masked template).
    
    The defaults are set to:
      - original_col: "original_text"
      - gold_col: "labeled_text"
      - pred_col: "masked_text"
    
    It then prints per-row metrics and aggregates overall evaluation metrics (Precision, Recall, F1, Accuracy).
    """
    # Read the file based on its extension.
    if file_path.endswith('.csv'):
        df = pd.read_csv(file_path)
    else:
        df = pd.read_pickle(file_path)
    
    # If the file is wrapped inside an object with a 'dataset' attribute, extract it.
    if hasattr(df, "dataset"):
        df = df.dataset

    # Print available columns for debugging.
    print("Available columns in the DataFrame:", df.columns.tolist())
    
    # Determine column names if not provided.
    if original_col is None:
        if "original_text" in df.columns:
            original_col = "original_text"
        else:
            raise ValueError("Could not determine original text column. Available columns: " + str(df.columns.tolist()))
    
    if gold_col is None:
        if "labeled_text" in df.columns:
            gold_col = "labeled_text"
        else:
            raise ValueError("Could not determine gold masked text column. Available columns: " + str(df.columns.tolist()))
    
    if pred_col is None:
        if "masked_text" in df.columns:
            pred_col = "masked_text"
        else:
            raise ValueError("Could not determine predicted masked text column. Available columns: " + str(df.columns.tolist()))
    
    # Verify the required columns exist.
    required_columns = [original_col, gold_col, pred_col]
    if not all(col in df.columns for col in required_columns):
        raise ValueError(f"DataFrame does not contain the required columns: {required_columns}")

    all_metrics = {"correct": 0, "misplaced": 0, "missing": 0, "false_positive": 0}
    all_examples = {"misplaced": []}
    detailed_results = []
    
    # Process each row.
    for idx, row in df.iterrows():
        metrics, examples = analyze_row_aligned(
            row[original_col],
            row[gold_col],
            row[pred_col]
        )
        detailed_results.append(metrics)
        for key in all_metrics:
            all_metrics[key] += metrics.get(key, 0)
        for ex in examples["misplaced"]:
            ex["row"] = idx
            ex["original"] = row[original_col]
            ex["labeled_text"] = row[gold_col]
            ex["masked_text"] = row[pred_col]
            all_examples["misplaced"].append(ex)
    
    metrics_df = pd.DataFrame(detailed_results)
    print("\nPer-row masking metrics:")
    print(metrics_df)
    
    print("\nAggregate masking metrics:")
    for key, value in all_metrics.items():
        print(f"{key}: {value}")
    
    note_counts = {"No valid label found at mapped span": 0, "Extra predicted token": 0}
    for ex in all_examples["misplaced"]:
        note = ex.get("note", "")
        if note in note_counts:
            note_counts[note] += 1

    print("\nCount of tokens with note 'No valid label found at mapped span':", 
          note_counts["No valid label found at mapped span"])
    print("Count of tokens with note 'Extra predicted token':", 
          note_counts["Extra predicted token"])
    
    print("\nExamples of Misplaced Tokens:")
    if all_examples["misplaced"]:
        for ex in all_examples["misplaced"]:
            print(f"Row {ex['row']} | Original Text:  {ex['original']}")
            print(f"Row {ex['row']} | Labeled Text:   {ex['labeled_text']}")
            print(f"Row {ex['row']} | Masked Text:    {ex['masked_text']}")
            print(f"  Note: {ex.get('note', '')}")
            print(f"  Gold: '{ex['gold']}' at {ex['gold_span']} vs Predicted: '{ex['predicted']}' at {ex['pred_span']}\n")
    else:
        print("No misplaced tokens found.")
    
    print(f"\nAmount of Misplaced Tokens: {len(all_examples['misplaced'])}")
    
    # Calculate evaluation metrics.
    TP = all_metrics["correct"]
    FN = all_metrics["missing"] + all_metrics["misplaced"]
    FP = all_metrics["false_positive"] + all_metrics["misplaced"]
    
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    F1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0
    
    print("\nEvaluation Metrics:")
    print(f"Precision: {precision:.3f}")
    print(f"Recall: {recall:.3f}")
    print(f"F1 Score: {F1:.3f}")
    print(f"Accuracy: {accuracy:.3f}")
    
    return all_metrics, metrics_df, all_examples

if __name__ == "__main__":
    # For a CSV file with columns: original_text, labeled_text, masked_text located in /tests,
    # the defaults will map as:
    #   original_col = "original_text"
    #   gold_col = "labeled_text"
    #   pred_col = "masked_text"
    aggregate_metrics, per_row_metrics, examples = analyze_masking("dataset_1.csv")

Available columns in the DataFrame: ['original_text', 'labeled_text', 'masked_text']

Per-row masking metrics:
      correct  misplaced  missing  false_positive
0           2          0        0               1
1           3          0        0               0
2           1          0        0               0
3           2          0        0               0
4           2          0        0               0
...       ...        ...      ...             ...
4265        1          0        0               0
4266        1          0        0               0
4267        4          0        0               0
4268        1          0        0               0
4269        4          0        0               0

[4270 rows x 4 columns]

Aggregate masking metrics:
correct: 8788
misplaced: 15
missing: 129
false_positive: 102

Count of tokens with note 'No valid label found at mapped span': 124
Count of tokens with note 'Extra predicted token': 102

Examples of Misplaced Tokens:
Row 0 | Original Te

In [70]:
#print  csv  masked_text, dataset_1.csv

#read csv file
df = pd.read_csv("dataset_1.csv")
#labeled_text
print(df.labeled_text[1])
print(df.masked_text[1])




write a research proposal for [ name ] [ name ] to study the social impact of immigration in [ location ] .
write a research proposal for [ name ] [ name ] to study the social impact of immigration in [ location ] .
